# Unidade II — Análise de Dados

## Similaridade e dissimilaridade

**Carga estimada:** 3 horas e 30 minutos  
**Pré-requisitos:** tipos de atributos, somatórios, vetores e raiz quadrada.

> **Pergunta norteadora:** quando dois objetos devem ser considerados próximos, e como a representação dos atributos altera essa resposta?


## Objetivos de aprendizagem

Ao concluir este notebook, você será capaz de:

- distinguir matriz de dados e matriz de dissimilaridades;
- calcular distâncias Euclidiana, Manhattan e Minkowski;
- calcular similaridade do cosseno e de Jaccard;
- escolher medidas para atributos numéricos, binários e mistos;
- explicar por que escala, codificação e dimensionalidade alteram vizinhanças.


## Proximidade e representação

Uma **similaridade** cresce quando objetos se parecem; uma **dissimilaridade** ou distância cresce quando diferem. A matriz de dados $\mathbf{X}$ possui objetos nas linhas e atributos nas colunas. Uma matriz de dissimilaridades $\mathbf{D}$ compara cada par de objetos, é quadrada e, para distâncias simétricas, satisfaz $D_{ij}=D_{ji}$ e $D_{ii}=0$.

Não existe medida universal. A escolha expressa o que significa estar próximo no problema e precisa respeitar a natureza dos atributos.


## Distâncias para atributos numéricos

Para $\mathbf{x},\mathbf{y}\in\mathbb{R}^d$, a distância de Minkowski de ordem $p\geq 1$ é

$$
d_p(\mathbf{x},\mathbf{y})=\left(\sum_{j=1}^{d}|x_j-y_j|^p\right)^{1/p}.
$$

Quando $p=1$, obtemos Manhattan; quando $p=2$, Euclidiana. Manhattan soma diferenças absolutas. Euclidiana enfatiza diferenças grandes por elevar ao quadrado antes da raiz.

Considere $\mathbf{x}=(1,2)$ e $\mathbf{y}=(4,6)$:

$$d_1=|1-4|+|2-6|=7,$$

$$d_2=\sqrt{(1-4)^2+(2-6)^2}=5.$$


In [1]:
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist, cosine, jaccard
from sklearn.preprocessing import StandardScaler

x = np.array([1.0, 2.0])
y = np.array([4.0, 6.0])

manhattan = np.abs(x - y).sum()
euclidiana = np.sqrt(np.sum((x - y) ** 2))
pd.Series({"Manhattan": manhattan, "Euclidiana": euclidiana})


Manhattan     7.0
Euclidiana    5.0
dtype: float64

### Matriz de dissimilaridades

Vamos calcular todas as distâncias entre quatro objetos. A diagonal deve ser zero e a matriz, simétrica.


In [2]:
objetos = pd.DataFrame(
    {"atributo_1": [1, 2, 5, 8], "atributo_2": [2, 2, 6, 9]},
    index=["A", "B", "C", "D"],
)
matriz_d = pd.DataFrame(
    cdist(objetos, objetos, metric="euclidean"),
    index=objetos.index, columns=objetos.index,
).round(2)
matriz_d


,A,B,C,D
A,0.00,1.00,5.66,9.90
B,1.00,0.00,5.00,9.22
C,5.66,5.00,0.00,4.24
D,9.90,9.22,4.24,0.00


## O efeito da escala

Se renda varia em milhares e idade em dezenas, a renda pode dominar uma distância, mesmo quando não deveria dominar o conceito de semelhança. Padronizar transforma cada atributo numérico pela média e desvio-padrão:

$$z_{ij}=\frac{x_{ij}-\bar{x}_j}{s_j}.$$

A transformação não é automaticamente correta: ela atribui escala comparável, mas a relevância substantiva de cada atributo ainda precisa ser decidida.


In [3]:
pessoas = pd.DataFrame(
    {"idade": [20, 22, 55], "renda_mensal": [2000, 8000, 8500]},
    index=["P1", "P2", "P3"],
)
sem_escala = pd.DataFrame(cdist(pessoas, pessoas), index=pessoas.index, columns=pessoas.index)
padronizados = StandardScaler().fit_transform(pessoas)
com_escala = pd.DataFrame(cdist(padronizados, padronizados), index=pessoas.index, columns=pessoas.index)

pd.concat({"sem padronização": sem_escala, "padronizada": com_escala}).round(2)


P1       P2       P3
sem padronização P1     0.00  6000.00  6500.09
                 P2  6000.00     0.00   501.09
                 P3  6500.09   501.09     0.00
padronizada      P1     0.00     2.04     3.10
                 P2     2.04     0.00     2.06
                 P3     3.10     2.06     0.00

## Similaridade do cosseno

Para vetores não nulos,

$$
\operatorname{sim}_{\cos}(\mathbf{x},\mathbf{y})=\frac{\mathbf{x}^T\mathbf{y}}{\|\mathbf{x}\|_2\|\mathbf{y}\|_2}.
$$

Ela mede o ângulo, sendo útil quando a direção do perfil importa mais que a magnitude, como em vetores de frequência de termos. Vetores proporcionais têm cosseno 1, ainda que suas magnitudes sejam diferentes.


In [4]:
documento_a = np.array([2.0, 1.0, 0.0])
documento_b = np.array([4.0, 2.0, 0.0])
documento_c = np.array([0.0, 1.0, 3.0])

pd.Series({
    "sim(A, B)": 1 - cosine(documento_a, documento_b),
    "sim(A, C)": 1 - cosine(documento_a, documento_c),
}).round(3)


sim(A, B)    1.000
sim(A, C)    0.141
dtype: float64

## Atributos binários e Jaccard

Para conjuntos $A$ e $B$, a similaridade de Jaccard é

$$J(A,B)=\frac{|A\cap B|}{|A\cup B|}.$$

Em vetores binários assimétricos, zeros conjuntos não contam: a ausência simultânea de milhares de itens não deveria tornar duas cestas artificialmente semelhantes. A distância é $1-J$.


In [5]:
cesta_a = np.array([1, 1, 0, 0, 1], dtype=bool)
cesta_b = np.array([1, 0, 1, 0, 1], dtype=bool)
intersecao = np.logical_and(cesta_a, cesta_b).sum()
uniao = np.logical_or(cesta_a, cesta_b).sum()

pd.Series({
    "Jaccard manual": intersecao / uniao,
    "Jaccard via SciPy": 1 - jaccard(cesta_a, cesta_b),
}).round(3)


Jaccard manual       0.5
Jaccard via SciPy    0.5
dtype: float64

## Atributos mistos e alta dimensionalidade

Dados reais combinam números, categorias, ordens e indicadores binários. Converter categorias arbitrariamente em inteiros e aplicar Euclidiana cria ordens e intervalos falsos. Uma alternativa é calcular contribuições adequadas por atributo, normalizá-las e agregá-las, como na dissimilaridade de Gower. Valores ausentes e pesos também precisam ser tratados explicitamente.

Em muitas dimensões, distâncias podem se concentrar: o vizinho mais próximo e o mais distante tornam-se relativamente parecidos. Seleção de atributos, redução de dimensionalidade e medidas específicas do domínio podem ser necessárias.

> **Verifique seu entendimento:** duas cestas compartilham 2 itens e possuem 5 itens distintos na união. Qual é a similaridade de Jaccard e qual é a distância correspondente?

> **Exercício:** crie quatro perfis com idade, renda, escolaridade ordinal e três preferências binárias. Proponha uma medida de dissimilaridade, explicando como cada tipo será tratado, se haverá padronização e que significado terá uma distância pequena. Compare ao menos dois pares manualmente.


## Síntese

- A proximidade depende da representação, do tipo dos atributos e do problema.
- Manhattan e Euclidiana são casos da distância de Minkowski.
- Escalas diferentes podem dominar distâncias numéricas.
- Cosseno compara direção; Jaccard compara presença compartilhada.
- Atributos mistos exigem contribuições compatíveis, não códigos numéricos arbitrários.

## Referências

- HAN, Jiawei; PEI, Jian; TONG, Hanghang. *Data Mining: Concepts and Techniques*. 4. ed. Cambridge: Morgan Kaufmann/Elsevier, 2023. Cap. 2, seção 2.3.
- GOWER, J. C. A general coefficient of similarity and some of its properties. *Biometrics*, v. 27, n. 4, p. 857–871, 1971.
